# gNFW Results Viewer

Load saved fitting results and plot corner diagrams, the gamma
posterior, best-fit model cube, and parameter summary.  No fitting
is performed — this notebook only reads the `result.npz` output
from `run_kgas_full.py`.

In [ ]:
from pathlib import Path
import tempfile
import os

import corner
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits as pyfits
from pymakeplots import pymakeplots as pmp

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})

## Configuration

In [ ]:
RESULTS_DIR = Path("results/KILOGAS007")

CELLSIZE = 0.2   # arcsec/pixel (must match fitting run)
NX = NY = 256    # spatial pixels
VSYS = 13583.0   # systemic velocity, km/s

## Load results

In [ ]:
res = np.load(RESULTS_DIR / "result.npz")

param_names = list(res["param_names"])
n_params = len(param_names)
best_params = dict(zip(param_names, res["params"]))

print(f"Parameters: {param_names}")
print(f"MAP values: {best_params}")
print(f"chi2: {float(res['chi2']):.2f}")
print(f"Reduced chi2: {float(res['reduced_chi2']):.6f}")

if "vmax" in res:
    print(f"Vmax (fixed): {float(res['vmax']):.1f} km/s")
if "r_scale" in res:
    print(f"r_scale (fixed): {float(res['r_scale']):.1f} arcsec")

## Convergence diagnostics

In [ ]:
if "converged" in res:
    print(f"Converged: {bool(res['converged'])}")
else:
    print("(Fixed-step run, no convergence info)")

if "autocorr_time" in res:
    tau = res["autocorr_time"]
    print(f"Autocorrelation time (per param):")
    for name, t in zip(param_names, tau):
        print(f"  {name:>12s}: {t:.1f} steps")
    print(f"  max(tau) = {np.max(tau):.1f}")

## Corner plot

In [ ]:
samples = res["chains"].reshape(-1, n_params)
labels = ["inc", "PA", "flux", r"$v_{\rm sys}$", r"$\sigma_{\rm gas}$", r"$\gamma$"]

fig = corner.corner(
    samples, labels=labels, show_titles=True,
    title_fmt=".3f",
)
fig.suptitle("gNFW posterior", y=1.02, fontsize=14)
plt.show()

## Gamma posterior

In [ ]:
gamma_idx = param_names.index("gamma")
gamma_samples = samples[:, gamma_idx]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(gamma_samples, bins=50, density=True, color="C0", alpha=0.7)
ax.axvline(0.0, color="C2", ls="--", lw=2, label=r"$\gamma=0$ (core)")
ax.axvline(1.0, color="C3", ls="--", lw=2, label=r"$\gamma=1$ (NFW cusp)")
med = np.median(gamma_samples)
lo, hi = np.percentile(gamma_samples, [16, 84])
ax.axvline(med, color="k", ls="-", lw=1.5,
           label=f"median = {med:.2f}")
ax.axvspan(lo, hi, alpha=0.15, color="k",
           label=f"68% CI = [{lo:.2f}, {hi:.2f}]")
ax.set_xlabel(r"$\gamma$ (inner density slope)")
ax.set_ylabel("Posterior density")
ax.legend()
ax.set_title("Inner slope posterior")
plt.tight_layout()
plt.show()

## Best-fit model cube (pymakeplots)

In [ ]:
def model_cube_to_fits(cube_vyx, vel_kms, cellsize_arcsec, filepath):
    """Write a (v,y,x) numpy model cube to a minimal FITS for pymakeplots."""
    nv, ny, nx = cube_vyx.shape
    dv = float(np.median(np.diff(vel_kms)))
    hdr = pyfits.Header()
    hdr['CTYPE1'] = 'RA---SIN'
    hdr['CTYPE2'] = 'DEC--SIN'
    hdr['CTYPE3'] = 'VRAD'
    hdr['CDELT1'] = -cellsize_arcsec / 3600.0
    hdr['CDELT2'] = cellsize_arcsec / 3600.0
    hdr['CDELT3'] = dv * 1000.0
    hdr['CRPIX1'] = nx / 2.0 + 0.5
    hdr['CRPIX2'] = ny / 2.0 + 0.5
    hdr['CRPIX3'] = 1.0
    hdr['CRVAL1'] = 180.0
    hdr['CRVAL2'] = 45.0
    hdr['CRVAL3'] = float(vel_kms[0]) * 1000.0
    hdr['CUNIT1'] = 'deg'
    hdr['CUNIT2'] = 'deg'
    hdr['CUNIT3'] = 'm/s'
    hdr['BMAJ'] = cellsize_arcsec / 3600.0
    hdr['BMIN'] = cellsize_arcsec / 3600.0
    hdr['BPA'] = 0.0
    hdr['BUNIT'] = 'Jy/beam'
    hdr['RESTFRQ'] = 230538000000.0
    pyfits.writeto(filepath, cube_vyx.astype(np.float32), hdr, overwrite=True)


cube_data = np.load(RESULTS_DIR / "bestfit_cube.npz")
cube_vyx = cube_data["cube"]
vel = cube_data["vel"]

tmpfile = tempfile.NamedTemporaryFile(suffix='.fits', delete=False)
model_cube_to_fits(cube_vyx, vel, CELLSIZE, tmpfile.name)
tmpfile.close()
try:
    p = pmp(cube=tmpfile.name)
    p.vsys = VSYS
    p.posang = best_params["pa"]
    gamma_map = best_params["gamma"]
    p.galname = f"gNFW ($\\gamma$={gamma_map:.2f})"
    p.make_all()
finally:
    os.unlink(tmpfile.name)

## Parameter summary table

In [ ]:
print(f"{'Param':<12} {'MAP':>12} {'Median':>12} {'16-84%':>20}")
print("-" * 60)
for i, name in enumerate(param_names):
    m = best_params[name]
    q = np.percentile(samples[:, i], [16, 50, 84])
    print(f"{name:<12} {m:>12.3f} {q[1]:>12.3f} [{q[0]:.3f}, {q[2]:.3f}]")